# AgentCore Observability for a Managed Knowledge Base

**What metrics can you pull from a Bedrock *Managed* Knowledge Base, and how do they surface once the KB is plugged into AgentCore?**

This notebook is the observability deep-dive for the Managed KB samples. It creates a KB + AgentCore Gateway (compactly — the [end-to-end notebook](../03-use-case-example/01-end-to-end-example-with-ac-gateway/01-bmkb-with-agentcore-gateway.ipynb) walks the setup in full), drives some traffic through it, and then spends its depth on **capturing, reading, and interpreting** every telemetry signal the KB and Gateway emit.

> **The headline** — from the [AWS docs](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-managed-observability.html): *AgentCore observability integration is available **only** for managed knowledge bases.* Once you enable runtime metrics and trace delivery, the AgentCore GenAI Observability page **auto-populates** your KB metrics and traces — no extra configuration. That is the whole "plug a Managed KB into AgentCore" story in one sentence.

## The 7-layer metric taxonomy

Every signal a Managed-KB RAG application can produce, grouped into layers. **This notebook covers Layers 1–5** (operational telemetry). The [RAG-evaluation notebook](../02-feature-examples/04-rag-evaluation/03-agentcore-evaluation-for-managed-kb.ipynb) is cumulative — it replays Layers 1–5 and adds **Layer 6** (tokens / cost) and **Layer 7** (quality scores).

| # | Layer | What it captures | Source | Auto or derived? | Here? |
|---|-------|------------------|--------|------------------|-------|
| 1 | **KB-native metrics** | Invocations, ClientErrors, ServerErrors, Throttles, TotalIterationCount, RawDataSize | `AWS/Bedrock/KnowledgeBases` | Auto | ✅ |
| 2 | **KB ingestion logs** | Per-doc crawl/sync/index status, `chunk_statistics` | `APPLICATION_LOGS` | Auto (log-only) | ✅ |
| 3 | **Retrieval quality** ⭐ | Chunk count, relevance-score stats | `Retrieve` response body | **Derived — nothing emits these** | ✅ |
| 4 | **Gateway / MCP** | Invocations, Latency, SystemErrors, UserErrors, Throttles | `AWS/Bedrock-AgentCore` | Auto | ✅ |
| 5 | **Traces (spans)** | Per-op latency, span tree, Gateway overhead | KB→X-Ray (`Retrieve`) + Gateway→`aws/spans` | Auto (needs delivery) | ✅ |
| 6 | RAG-app | Input/output tokens, cost $, latency split | Response / agent result | Derived | → eval nb |
| 7 | Quality / eval | Faithfulness, Correctness, ToolSelectionAccuracy | AgentCore Evaluate | LLM-as-judge | → eval nb |

> ⚠️ **There is no `Latency` metric** in `AWS/Bedrock/KnowledgeBases`. KB request latency lives in **X-Ray traces (Layer 5)**, not CloudWatch metrics. We return to this in Layer 1.

## Where every signal originates and lands

```
                          ┌─── Layer 1: AWS/Bedrock/KnowledgeBases   (metrics)
  S3 ──▶ Managed KB ──────┼─── Layer 2: APPLICATION_LOGS + job stats  (ingestion)
                          └─── Layer 5a: TRACES ──▶ X-Ray            (Retrieve only)
        ▲
        │ Retrieve / AgenticRetrieveStream
        │
  Strands Agent ──MCP──▶ AgentCore Gateway ───┬── Layer 4: AWS/Bedrock-AgentCore  (metrics)
        │                                     └── Layer 5b: OTEL spans ──▶ aws/spans
        ▼
  Retrieve response body ──▶ Layer 3: retrieval quality (DERIVED → put_metric_data → CloudWatch)
```

Notice **no agent deployment**. The Gateway emits its spans (Layer 5b) server-side, and the KB emits metrics/traces server-side — so a Strands agent running locally in this notebook produces the full telemetry set. Gateway spans correlate by **`gateway.id` + `traceId`** (not `session.id`, which only an AgentCore Runtime stamps — see Layer 5).

## Prerequisites

- AWS credentials with Bedrock, IAM, AgentCore, CloudWatch, and X-Ray permissions
- Model access enabled for a generation model (Claude) — managed embedding is the default
- Python 3.10+; **CloudWatch Transaction Search** is enabled below (needed for spans)

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../requirements.txt --quiet
%pip install strands-agents strands-agents-tools mcp-proxy-for-aws jinja2 --quiet

In [ ]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

## Step 1 — Configuration

Import the shared `kb_observability` helpers and set names. If you ran the end-to-end notebook, `%store -r kb_id` restores that KB; otherwise this notebook creates one.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, boto3, sys, time, uuid, json
import pandas as pd

# Credentials: force the isengard-backed profile. Hard-assign (NOT setdefault) —
# an IDE kernel may already carry an AWS_PROFILE that setdefault would not override.

os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'

sys.path.insert(0, '..')
from utils.managed_knowledge_base import ManagedKnowledgeBase
from utils import kb_observability as obs
from utils import nb_display as ui   # ANSI status output + pandas Styler helpers

# Kernel sanity check — this MUST point at bmkb-code-samples/.venv, not ~/Documents/.venv.
# A fresh boto3 Session picks up AWS_PROFILE set just above (a client cached earlier would not).
# print('Python:', sys.executable)
region = boto3.session.Session().region_name or os.environ['AWS_DEFAULT_REGION']
account_id = boto3.session.Session().client('sts').get_caller_identity()['Account']
suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]

region_prefix = next((v for k, v in {'us-':'us','eu-':'eu','ap-':'apac'}.items() if region.startswith(k)), 'us')
generation_model_arn = f'arn:aws:bedrock:{region}:{account_id}:inference-profile/{region_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'

# session.id — the correlation key threaded through every read below
session_id = f'bmkb-obs-{uuid.uuid4().hex[:8]}'

kb_name = f'bmkb-obs-{suffix}'
bucket_name = f'bedrock-bmkb-obs-{suffix}-{account_id}'
gateway_name = f'bmkb-obs-gw-{suffix}'

ui.rule('Configuration')
ui.ok(f'Region: {region}   Account: {account_id}')
ui.ok(f'Session ID: {session_id}')
ui.info('(session.id tags our custom retrieval-quality metrics; Gateway spans correlate by gateway.id + traceId)')

## Step 2 — Create (or restore) the KB and Gateway

We compress the end-to-end notebook's setup into a few calls. See that notebook for the annotated walkthrough of IAM roles, the Gateway, and the KB target.

In [ ]:
s3 = boto3.client('s3')

# Restore a previously-created KB id if present.
try:
    kb_id  # defined if a prior cell set it this session
except NameError:
    try:
        %store -r kb_id
    except Exception:
        pass
kb_id = kb_id if 'kb_id' in dir() else None

# Validate the stored KB still exists — a stale %store id (e.g. after cleanup) would
# otherwise cause ResourceNotFoundException later. Recreate if missing.
if kb_id:
    try:
        boto3.client('bedrock-agent', region_name=region).get_knowledge_base(knowledgeBaseId=kb_id)
        print(f'Reusing stored KB: {kb_id}')
        kb = ManagedKnowledgeBase.from_existing(kb_id, region_name=region)
    except Exception:
        print(f'Stored KB {kb_id} no longer exists — creating a fresh one.')
        kb_id = None

if not kb_id:
    if region == 'us-east-1':
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={'LocationConstraint': region})
    s3.upload_file('../synthetic_dataset/octank_financial_10K.pdf', bucket_name, 'octank_financial_10K.pdf')

    kb = ManagedKnowledgeBase(
        kb_name=kb_name, bucket_name=bucket_name,
        enable_logging=True, region_name=region, suffix=suffix,
    )
    kb_id = kb.kb_id
    %store kb_id
    time.sleep(30)
    kb.start_ingestion_job()

print(f'KB ID: {kb_id}')

## Step 3 — Create the Gateway and enable all telemetry delivery *up front*

Metrics are automatic, but **spans and vended logs must be delivered before the calls you want to observe**. We enable the full delivery chain now:

```
PutDeliverySource ──▶ PutDeliveryDestination ──▶ CreateDelivery
   (what to ship)        (where it goes: CWL / X-Ray)     (connect them)
```

- **KB traces (Layer 5a)** → X-Ray, emitted for the `Retrieve` operation only
- **Gateway spans (Layer 5b)** → `aws/spans`
- **CloudWatch Transaction Search** → the prerequisite that lets X-Ray write spans as structured logs

These calls are **idempotent** — re-running the cell is safe (we swallow `ConflictException` / already-exists).

In [ ]:
# Guard: if the kernel holds a stale `kb` (e.g. from an earlier run), rebuild the handle.
if not hasattr(kb, '_session'):
    kb = ManagedKnowledgeBase.from_existing(kb_id, region_name=region)

ac = kb.get_agentcore_client()

# Reuse a previously-created Gateway if one is stored and still exists — otherwise a
# fresh Gateway would be created on every run (they don't self-clean and count against
# the account quota). Mirrors the %store kb_id pattern above.
try:
    %store -r gateway_id
except Exception:
    pass

gateway_id = gateway_id if 'gateway_id' in dir() else None
gateway_url = None
if gateway_id:
    try:
        g = ac.get_gateway(gatewayIdentifier=gateway_id)
        gateway_url = g['gatewayUrl']
        print(f'Reusing Gateway: {gateway_id} ({g["status"]})')
    except Exception:
        gateway_id = None  # stored id no longer exists → recreate below

if not gateway_id:
    gw_role_name = f'AmazonBedrockGatewayRole_{suffix}'
    iam = boto3.client('iam')
    try:
        role = iam.create_role(
            RoleName=gw_role_name,
            AssumeRolePolicyDocument=json.dumps({'Version':'2012-10-17','Statement':[{
                'Effect':'Allow','Principal':{'Service':'bedrock-agentcore.amazonaws.com'},
                'Action':'sts:AssumeRole','Condition':{'StringEquals':{'aws:SourceAccount':account_id}}}]}),
        )
    except iam.exceptions.EntityAlreadyExistsException:
        role = iam.get_role(RoleName=gw_role_name)
    iam.put_role_policy(RoleName=gw_role_name, PolicyName='kb-retrieve',
        PolicyDocument=json.dumps({'Version':'2012-10-17','Statement':[{'Effect':'Allow',
            'Action':['bedrock:Retrieve','bedrock:GetKnowledgeBase','bedrock:ListKnowledgeBases'],
            'Resource':f'arn:aws:bedrock:{region}:{account_id}:knowledge-base/*'}]}))
    gateway_role_arn = role['Role']['Arn']
    time.sleep(20)

    gw = kb.create_gateway(gateway_name=gateway_name, gateway_role_arn=gateway_role_arn, auth_type='AWS_IAM')
    gateway_id, gateway_url = gw['gateway_id'], gw['gateway_url']
    kb.create_gateway_kb_target(gateway_id=gateway_id, target_name='kb-retrieve', num_results=5)
    %store gateway_id
else:
    gw_role_name = f'AmazonBedrockGatewayRole_{suffix}'  # for cleanup reference
    iam = boto3.client('iam')

print(f'Gateway: {gateway_id}')

In [ ]:
# Enable delivery: Transaction Search + KB traces (X-Ray) + Gateway spans/logs.
logs = boto3.client('logs', region_name=region)
xray = boto3.client('xray', region_name=region)

def _ok(fn, label):
    try:
        r = fn(); print(f'  ✓ {label}'); return r
    except Exception as e:
        if any(k in str(e).lower() for k in ['already', 'conflict']):
            print(f'  ✓ {label} (already configured)')
        else:
            print(f'  ✗ {label}: {e}')

# A — CloudWatch Transaction Search (spans as structured logs)
_ok(lambda: logs.put_resource_policy(policyName='AgentCoreTransactionSearchAccess',
    policyDocument=json.dumps({'Version':'2012-10-17','Statement':[{'Sid':'TxnSearch','Effect':'Allow',
        'Principal':{'Service':'xray.amazonaws.com'},'Action':'logs:PutLogEvents',
        'Resource':[f'arn:aws:logs:{region}:{account_id}:log-group:aws/spans:*',
                    f'arn:aws:logs:{region}:{account_id}:log-group:/aws/application-signals/data:*'],
        'Condition':{'ArnLike':{'aws:SourceArn':f'arn:aws:xray:{region}:{account_id}:*'},
                     'StringEquals':{'aws:SourceAccount':account_id}}}]})), 'Transaction Search policy')
_ok(lambda: xray.update_trace_segment_destination(Destination='CloudWatchLogs'), 'Traces → CloudWatch Logs')
# Sample 90% — the DEFAULT is 1%, which drops nearly all spans for low-volume demos.
# (Account-level setting; 90% keeps the demo observable without indexing literally everything.)
_ok(lambda: xray.update_indexing_rule(Name='Default',
    Rule={'Probabilistic': {'DesiredSamplingPercentage': 90}}), 'Span sampling → 90%')

# B — KB traces → X-Ray (Retrieve op only)
kb_arn = f'arn:aws:bedrock:{region}:{account_id}:knowledge-base/{kb_id}'
_ok(lambda: logs.put_delivery_source(name=f'kb-{kb_id}-traces', logType='TRACES', resourceArn=kb_arn), 'KB traces source')
dest = _ok(lambda: logs.put_delivery_destination(name=f'kb-{kb_id}-xray', deliveryDestinationType='XRAY'), 'KB traces → X-Ray')
if dest:
    _ok(lambda: logs.create_delivery(deliverySourceName=f'kb-{kb_id}-traces',
        deliveryDestinationArn=dest['deliveryDestination']['arn']), 'KB traces delivery')

# C — Gateway spans → aws/spans
gw_arn = f'arn:aws:bedrock-agentcore:{region}:{account_id}:gateway/{gateway_id}'
_ok(lambda: logs.put_delivery_source(name=f'gw-{gateway_id}-traces', logType='TRACES', resourceArn=gw_arn), 'Gateway traces source')
gdest = _ok(lambda: logs.put_delivery_destination(name=f'gw-{gateway_id}-xray', deliveryDestinationType='XRAY'), 'Gateway traces → X-Ray')
if gdest:
    _ok(lambda: logs.create_delivery(deliverySourceName=f'gw-{gateway_id}-traces',
        deliveryDestinationArn=gdest['deliveryDestination']['arn']), 'Gateway traces delivery')
print('\nDelivery configured. Subsequent calls are observable.')

## Step 4 — Generate traffic (with `session.id` attached)

We run a few queries two ways so every layer has data:
- **Direct `Retrieve`** — populates Layer 1 metrics + Layer 5a KB→X-Ray traces (Retrieve-only) + gives us response bodies for Layer 3.
- **Strands agent → Gateway** — populates Layer 4 Gateway metrics + Layer 5b `aws/spans`.

The Gateway emits its spans server-side, so the agent runs **locally right here** — no deployment.

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel
from strands.tools.mcp import MCPClient
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client
from opentelemetry import baggage, context

queries = [
    "What is Octank Financial's total revenue?",
    "What are Octank's main risk factors?",
    "Describe Octank's growth strategy.",
]

# Direct Retrieve — keep the responses for Layer 3
retrieve_responses = []
for q in queries:
    resp = kb.retrieve(q, num_results=5) if hasattr(kb, 'retrieve') else \
           boto3.client('bedrock-agent-runtime', region_name=region).retrieve(
               knowledgeBaseId=kb_id, retrievalQuery={'text': q},
               retrievalConfiguration={'managedSearchConfiguration': {'numberOfResults': 5}})
    retrieve_responses.append(resp)
print(f'{len(retrieve_responses)} Retrieve calls done.')

# Strands agent → Gateway, with session.id in OTEL baggage (the join key)
mcp_client = MCPClient(lambda: aws_iam_streamablehttp_client(
    endpoint=gateway_url, aws_region=region, aws_service='bedrock-agentcore'))
model = BedrockModel(model_id=f'{region_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0', region_name=region)

token = context.attach(baggage.set_baggage('session.id', session_id))
try:
    with mcp_client:
        agent = Agent(model=model, tools=mcp_client.list_tools_sync(),
                      system_prompt='Answer using the knowledge base tool. Cite sources.')
        for q in queries:
            agent(q)
finally:
    context.detach(token)  # MUST detach or baggage leaks across requests
print('Agent→Gateway calls done. Waiting 60s for span/metric ingestion...')
time.sleep(60)

## Layer 1 — KB-native metrics

Amazon Bedrock publishes these to `AWS/Bedrock/KnowledgeBases` **automatically**, at no cost, for every request.

| Metric | Unit | Notes |
|--------|------|-------|
| `Invocations` | Count | every request, incl. errors |
| `ClientErrors` | Count | 4xx (non-throttle), only when it occurs |
| `ServerErrors` | Count | 5xx, only when it occurs |
| `Throttles` | Count | 429, only when it occurs |
| `TotalIterationCount` | Count | **AgenticRetrieveStream only** — agentic iterations |
| `RawDataSize` | GB | storage; published after an ingestion job completes |

Dimensions: `Operation` (`Retrieve` \| `AgenticRetrieveStream`), `KnowledgeBaseId`.

> ⚠️ **No `Latency` metric exists here.** A common mistake is to read `Latency` from this namespace — it returns nothing. Latency is in **X-Ray traces (Layer 5)**. We prove this in Layer 5.

> **Why a clean run shows only `Invocations`:** the error/throttle metrics are **published only when they occur** — a healthy run emits none, so their absence is success, not a gap. `TotalIterationCount` appears only for `AgenticRetrieveStream` (we use `Retrieve` here). `RawDataSize` is published only *after* an ingestion job completes — reusing an existing KB means no new ingestion, so it may not appear this run.

In [ ]:
df = obs.get_kb_metrics(kb_id, region_name=region, hours=1)
ui.rule('Layer 1 — KB-native metrics')
ui.ok(f"Latency metric present? {'Latency' in df['metric'].values}  (expected: False — KB latency lives in traces)")
display(ui.style_metrics(df))

## Layer 2 — KB ingestion observability

Two complementary views of how documents made it into the KB:

**A. Ingestion-job statistics (authoritative aggregate)** — from the `get_ingestion_job` API: documents scanned / indexed / failed / skipped. This is the reliable "did my corpus ingest cleanly?" signal, and the dependable source for chunk/document counts.

**B. Per-document logs (detail)** — from `APPLICATION_LOGS` (enabled by `enable_logging=True`): each file's crawl → sync → index status.

```
Ingestion job ──▶ statistics{scanned, indexed, failed, skipped}          ← aggregate (API)
   └─ per document ──▶ crawl_status ─▶ sync_status ─▶ index_status         ← detail (logs)
```

> **On `chunk_statistics`:** the docs describe a per-document `chunk_statistics` field, but it is **not emitted by every KB/connector** (it was absent on this run — even for a fresh `ADD` ingestion). So we lead with the job `statistics`, which are always present, and treat the per-doc `chunk_statistics` column as opportunistic.

In [ ]:
ui.rule('Layer 2 — KB ingestion observability')

# A — authoritative aggregate from the ingestion-job API
ui.ok('Ingestion job statistics (documents scanned / indexed / failed / skipped):')
display(obs.get_ingestion_stats(kb_id, region_name=region))

# B — per-document detail from APPLICATION_LOGS
rows = obs.query_ingestion_logs(kb_id, region_name=region, hours=24)
if not rows:
    ui.info('No per-document logs yet (ingestion still running, or logs still landing).')
else:
    flat = [{f['field']: f['value'] for f in row if f['field'] != '@ptr'} for row in rows]
    ui.ok('Per-document ingestion status (crawl → sync → index):')
    display(pd.DataFrame(flat))

## Layer 3 — Retrieval quality ⭐ (the signature KB signal)

Every `Retrieve` response carries a **relevance score** and **chunk provenance** per result — but **nothing captures them**. No CloudWatch metric, no log. Retrieval *quality* is invisible unless you derive it.

Each result includes `metadata._chunk_id` and `_document_title`/`_source_uri` — so beyond scores we can measure **coverage**: how many distinct source documents the results span. (This is *retrieval-time* chunk info, distinct from the *ingestion-time* `chunk_statistics` in Layer 2.)

We do two things:
1. **Extract** chunk count, distinct-doc coverage, and score stats (min / mean / max / spread) from the response body.
2. **Publish** them as **CloudWatch custom metrics** (`put_metric_data`) into the `BMKB/RetrievalQuality` namespace, so they become graphable and alarmable next to the native metrics.

```
Retrieve response ─▶ extract_retrieval_quality() ─▶ emit_retrieval_metrics() ─▶ BMKB/RetrievalQuality
```

Reading the signals: score **spread** — a tight, high cluster means confident retrieval; a wide spread means the KB is mixing strong and weak matches. **distinct_docs** — many results from one document is narrow coverage; results spread across documents is broad. Both are early signals to tune chunking, top-k, or reranking.

In [ ]:
quality = [obs.extract_retrieval_quality(r) for r in retrieve_responses]
quality_df = pd.DataFrame(quality)
quality_df.insert(0, 'query', [q[:40] for q in queries])

ui.rule('Layer 3 — Retrieval quality')
display(ui.style_scores(quality_df))

# Publish each to CloudWatch custom metrics (BMKB/RetrievalQuality namespace).
for stats in quality:
    obs.emit_retrieval_metrics(stats, kb_id=kb_id, region_name=region)
ui.ok(f'Published {len(quality)} retrieval-quality records to {obs.RETRIEVAL_QUALITY_NAMESPACE}')

## Layer 4 — Gateway / MCP metrics

The AgentCore Gateway publishes these to `AWS/Bedrock-AgentCore` automatically. Unlike the KB namespace, this one **does** include `Latency` (the Gateway measures request duration itself).

In [ ]:
ui.rule('Layer 4 — Gateway / MCP metrics')
display(ui.style_metrics(obs.get_gateway_metrics(region_name=region, hours=1)))

## Layer 5 — Traces (spans)

Two independent span sources:

```
5a  KB ──▶ X-Ray            (Retrieve op only; view in the X-Ray / CloudWatch console)
5b  Gateway ──▶ aws/spans     (every MCP op; queryable via Logs Insights below)
```

We query `aws/spans` filtered by **`gateway.id`** — the attribute the Gateway stamps on every span it emits. Each MCP *Call Tool* produces two spans: `kind=SERVER` (`AgentCore.Gateway.InvokeTool` — the overall Gateway handling) and `kind=CLIENT` (`...InvokeTool.kb-retrieve___Retrieve` — the KB target call). The SERVER span even reports **`overhead_latency_ms`** directly — the Gateway's own processing time on top of the downstream KB call.

> **On `session.id`:** Gateway spans do **not** carry a `session.id` — that field only appears on spans emitted by an AgentCore **Runtime** invoked with the `X-Amzn-Bedrock-AgentCore-Runtime-Session-Id` header. Our agent runs **locally** and calls the Gateway directly, so there is no runtime to stamp it. Here you correlate by `gateway.id` + `traceId` (which groups the spans of one request). The `session.id`-as-join-key pattern returns in the eval notebook, where the agent's own instrumented spans carry it.

In [ ]:
ui.rule('Layer 5 — Traces (spans)')
spans = obs.query_spans(region_name=region, gateway_id=gateway_id, hours=1)
if not spans:
    ui.info('No spans yet — they can take 1–2 min to land in aws/spans after the calls. Re-run this cell.')

rows = []
for s in spans:
    a = s.get('attributes', {})
    rows.append({'name': s.get('name', '?'), 'kind': s.get('kind', ''),
                 'duration_ms': round(s.get('durationNano', 0) / 1_000_000, 1),
                 'latency_ms': a.get('latency_ms', ''),
                 'overhead_ms': a.get('overhead_latency_ms', ''),
                 'trace_id': s.get('traceId', '')[:16]})
display(pd.DataFrame(rows))

# The Gateway reports its own overhead directly on the SERVER span — no math needed.
overheads = [int(a['overhead_latency_ms']) for s in spans
             if (a := s.get('attributes', {})).get('overhead_latency_ms')]
if overheads:
    ui.ok(f'Gateway overhead: {min(overheads)}–{max(overheads)} ms across {len(overheads)} calls '
          f'(from attributes.overhead_latency_ms)')

## Step 5 — A CloudWatch dashboard focused on the Managed KB

One `put_dashboard` call unifies the layers into a single board — Layer 1 KB metrics, Layer 3 derived retrieval quality, Layer 4 Gateway metrics, and a Layer 5 span widget filtered by this run's `gateway.id`. This is the same shape the RAG-evaluation notebook extends with cost (Layer 6) and quality scores (Layer 7).

In [ ]:
dashboard_name = f'BMKB-Observability-{suffix}'
obs.build_kb_dashboard(dashboard_name, kb_id=kb_id, gateway_id=gateway_id, region_name=region)
ui.rule('CloudWatch dashboard')
ui.ok(f'Dashboard created: {dashboard_name}')
ui.info(f'https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#dashboards/dashboard/{dashboard_name}')

## The AgentCore console payoff

Because this is a **managed** KB, everything you just captured by hand also appears — with **no extra configuration** — in the AgentCore GenAI Observability page. The KB metrics and traces you enabled flow straight into the consolidated view:

**CloudWatch → GenAI Observability → Bedrock AgentCore**: <https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability>

Use the **All spans / All traces** tabs and filter by `gateway.id` (this run's gateway) to see the same Gateway spans we queried above, now correlated with the KB metrics in one place.

> **What you'll see empty — and why it's expected:** in the trace view, **"All Events (0)"** and **"Tokens: 0"** are normal for this notebook. Span *events* (the gen-AI prompt/response/tool-IO records) and token counts come from the **agent's own instrumented spans**, not the Gateway. Our agent runs **locally** and calls the Gateway directly, so only the Gateway's server-side spans reach `aws/spans` — you get the span tree, latencies, and overhead, but not events/tokens/session grouping.
>
> This draws the line cleanly: **the KB-plugged-into-AgentCore path gives you rich server-side telemetry for free**; **events, tokens, and session.id require instrumenting or deploying the agent** — the path the [RAG-evaluation notebook](../02-feature-examples/04-rag-evaluation/03-agentcore-evaluation-for-managed-kb.ipynb) takes.

## Summary — every signal, where it lives, how to capture it

| Layer | Signal | Where | How (this notebook) |
|-------|--------|-------|---------------------|
| 1 | KB metrics | `AWS/Bedrock/KnowledgeBases` | `obs.get_kb_metrics()` — **no Latency metric** |
| 2 | Ingestion counts + per-doc status | `get_ingestion_job` API + `APPLICATION_LOGS` | `obs.get_ingestion_stats()` + `obs.query_ingestion_logs()` |
| 3 | Retrieval quality ⭐ | derived → `BMKB/RetrievalQuality` | `obs.extract_retrieval_quality()` + `obs.emit_retrieval_metrics()` |
| 4 | Gateway metrics | `AWS/Bedrock-AgentCore` | `obs.get_gateway_metrics()` |
| 5 | Spans | KB→X-Ray + Gateway→`aws/spans` | `obs.query_spans(gateway_id=...)` |
| — | Everything, correlated | AgentCore GenAI Observability | auto — managed KB only |

**Next:** the [RAG-evaluation notebook](../02-feature-examples/04-rag-evaluation/03-agentcore-evaluation-for-managed-kb.ipynb) reuses these same helpers and adds cost (Layer 6) and quality scores (Layer 7).

## Cleanup

Deletes **everything this notebook created**, in dependency order:
Gateway + target → trace-delivery chains → KB + data sources + KB IAM role/policies + ingestion-log delivery → S3 bucket (the uploaded copy) → Gateway IAM role → dashboard → stored ids.

Commented out — uncomment and run deliberately.

> **Safety:** scoped to this run's `kb_id` / `gateway_id` / `suffix`, so it only removes resources this notebook created. The original `synthetic_dataset/octank_financial_10K.pdf` in the repo is never touched (only the S3 *copy* is deleted).

In [ ]:
# Cleanup — uncomment to run. Deletes EVERYTHING this notebook created; leaves nothing behind.
# Ordered: gateway → deliveries → KB → IAM → S3 → dashboard. Each step guarded so one
# failure doesn't block the rest. Only touches this run's resources (by kb_id / gateway_id / suffix).

# ui.rule('Cleanup')
# logs_c = boto3.client('logs', region_name=region)
#
# # 1. Gateway: target(s) then gateway
# for t in ac.list_gateway_targets(gatewayIdentifier=gateway_id).get('items', []):
#     ac.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=t['targetId'])
# time.sleep(5)
# ac.delete_gateway(gatewayIdentifier=gateway_id)
# ui.ok(f'Deleted gateway {gateway_id}')
#
# # 2. Trace-delivery chains created in Step 3 (KB→X-Ray, Gateway→X-Ray) — util doesn't cover these.
# for base in (f'kb-{kb_id}', f'gw-{gateway_id}'):
#     for d in logs_c.describe_deliveries().get('deliveries', []):
#         if d.get('deliverySourceName') in (f'{base}-traces',):
#             logs_c.delete_delivery(id=d['id'])
#     for nm, fn in [(f'{base}-traces', logs_c.delete_delivery_source),
#                    (f'{base}-xray', logs_c.delete_delivery_destination)]:
#         try: fn(name=nm)
#         except Exception: pass
# ui.ok('Deleted trace-delivery chains')
#
# # 3. KB + data sources + KB-IAM + S3 + ingestion-log delivery.
# #    Full object self-cleans everything; a from_existing handle can only delete the KB,
# #    so we remove the KB exec role, its policies, and the bucket explicitly.
# if hasattr(kb, '_data_sources'):
#     kb.delete_kb(delete_iam=True, delete_s3_bucket=True)
# else:
#     ManagedKnowledgeBase.delete_kb_by_id(kb_id, region_name=region)
#     kb_role = f'AmazonBedrockExecutionRoleForKnowledgeBase_{suffix}'
#     for pol in (f'AmazonBedrockCloudWatchPolicyForKnowledgeBase_{suffix}',
#                 f'AmazonBedrockS3PolicyForKnowledgeBase_{suffix}'):
#         arn = f'arn:aws:iam::{account_id}:policy/{pol}'
#         try: iam.detach_role_policy(RoleName=kb_role, PolicyArn=arn)
#         except Exception: pass
#         try: iam.delete_policy(PolicyArn=arn)
#         except Exception: pass
#     try: iam.delete_role(RoleName=kb_role)
#     except Exception as e: ui.info(f'KB role: {e}')
#     try:
#         b = boto3.resource('s3').Bucket(bucket_name)
#         b.objects.all().delete(); b.delete()
#     except Exception as e: ui.info(f'bucket: {e}')
# ui.ok('Deleted KB, KB IAM, and S3 bucket')
#
# # 4. Gateway IAM role + inline policy
# try:
#     iam.delete_role_policy(RoleName=gw_role_name, PolicyName='kb-retrieve')
#     iam.delete_role(RoleName=gw_role_name)
# except Exception as e:
#     ui.info(f'gateway role cleanup: {e}')
#
# # 5. Dashboard + clear stored ids
# boto3.client('cloudwatch', region_name=region).delete_dashboards(DashboardNames=[dashboard_name])
# %store -d kb_id
# %store -d gateway_id
# ui.ok('Cleanup complete — nothing left behind.')